In [5]:
import os
import posixpath
from pathlib import Path
import dropbox
import time
from dropbox.exceptions import ApiError, AuthError
from dropbox.files import FileMetadata, FolderMetadata

ACCESS_TOKEN = os.environ.get("DROPBOX_ACCESS_TOKEN", "")
if not ACCESS_TOKEN:
    ACCESS_TOKEN = input("Enter Dropbox access token (or set DROPBOX_ACCESS_TOKEN): ").strip()
if not ACCESS_TOKEN:
    raise ValueError("Dropbox access token is required. Set DROPBOX_ACCESS_TOKEN or provide one interactively.")

SHARED_LINK = "https://www.dropbox.com/scl/fo/p1xamlw94wt1p29de542e/h/?e=1"  # Your shared folder URL


def find_workspace_root():
    cwd = Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").exists():
            return parent
    return cwd


AUDIO_EXTENSIONS = {
    ".mp3", ".wav", ".flac", ".aac", ".ogg",
    ".m4a", ".wma", ".aiff", ".alac", ".opus"
}


dbx = dropbox.Dropbox(ACCESS_TOKEN)


def is_audio(filename):
    _, ext = os.path.splitext(filename)
    return ext.lower() in AUDIO_EXTENSIONS


def shared_link_path(path):
    if not path:
        return ""
    return path if path.startswith("/") else "/" + path


def verify_local_file(local_file, expected_size):
    if not os.path.exists(local_file):
        return False
    if expected_size is None:
        return True
    try:
        return os.path.getsize(local_file) == expected_size
    except OSError:
        return False


def _download_with_retries(shared_link_url, file_path, local_file, entry, max_retries=5, backoff=1):
    """Download a shared link file into local_file with retries and size verification.
    Returns True on success, False if skipped or failed after retries."""
    temp_file = local_file + ".part"
    for attempt in range(1, max_retries + 1):
        try:
            metadata, response = dbx.sharing_get_shared_link_file(url=shared_link_url, path=file_path)

            # write to a temp file first
            with open(temp_file, "wb") as f:
                f.write(response.content)

            expected_size = getattr(entry, "size", None)
            actual_size = None
            try:
                actual_size = os.path.getsize(temp_file)
            except Exception:
                pass

            if expected_size is not None and actual_size is not None and actual_size != expected_size:
                print(f"Size mismatch for {file_path}: expected={expected_size}, got={actual_size}")
                try:
                    os.remove(temp_file)
                except Exception:
                    pass
                if attempt == max_retries:
                    print(f"Max retries reached for {file_path} after size mismatch; skipping.")
                    return False
                time.sleep(backoff)
                backoff *= 2
                continue

            # move into place atomically
            os.replace(temp_file, local_file)
            return True

        except AuthError as exc:
            err_msg = str(exc)
            print("Dropbox auth error while downloading shared link file:", err_msg)
            if "expired_access_token" in err_msg or "invalid_access_token" in err_msg:
                print("Your Dropbox access token has expired or is invalid.")
                print("Set DROPBOX_ACCESS_TOKEN to a fresh access token and restart the notebook.")
            else:
                print("Dropbox token requires sharing.read scope for shared link file downloads.")
                print("Regenerate ACCESS_TOKEN with sharing.read and retry.")
            raise
        except ApiError as exc:
            if "shared_link_access_denied" in str(exc):
                print(f"Skipping file (access denied): {file_path}")
                try:
                    if os.path.exists(temp_file):
                        os.remove(temp_file)
                except Exception:
                    pass
                return False
            print(f"ApiError on attempt {attempt} for {file_path}: {exc}")
            if attempt == max_retries:
                raise
        except Exception as exc:
            print(f"Network error on attempt {attempt} for {file_path}: {exc}")
            if attempt == max_retries:
                raise
        time.sleep(backoff)
        backoff *= 2
    return False


def download_audio_from_shared_folder(shared_link_url, local_root):
    os.makedirs(local_root, exist_ok=True)
    shared_link = dropbox.files.SharedLink(url=shared_link_url)

    def process_folder(dropbox_path, local_path):
        print(f"Listing shared folder path: {repr(dropbox_path) or '/'}")
        try:
            result = dbx.files_list_folder(
                path=shared_link_path(dropbox_path),
                shared_link=shared_link
            )
        except ApiError as exc:
            print("Failed to list Dropbox shared folder path:", repr(dropbox_path), exc)
            raise

        while True:
            for entry in result.entries:
                if isinstance(entry, FolderMetadata):
                    subfolder_local = os.path.join(local_path, entry.name)
                    os.makedirs(subfolder_local, exist_ok=True)
                    next_path = entry.name if not dropbox_path else posixpath.join(dropbox_path, entry.name)
                    process_folder(next_path, subfolder_local)

                elif isinstance(entry, FileMetadata):
                    if not is_audio(entry.name):
                        continue

                    local_file = os.path.join(local_path, entry.name)
                    file_path = entry.name if not dropbox_path else posixpath.join(dropbox_path, entry.name)
                    file_path = shared_link_path(file_path)
                    expected_size = getattr(entry, "size", None)

                    if os.path.exists(local_file):
                        if verify_local_file(local_file, expected_size):
                            print(f"Verified existing file: {local_file}")
                            continue
                        print(f"Local file mismatch or incomplete, deleting and re-downloading: {local_file}")
                        try:
                            os.remove(local_file)
                        except OSError as e:
                            print(f"Warning: could not delete {local_file}: {e}")

                    print(f"Downloading: {file_path}")
                    ok = _download_with_retries(shared_link_url, file_path, local_file, entry)
                    if not ok:
                        print(f"Failed to download or verify: {file_path}")
                        continue

            if not result.has_more:
                break

            result = dbx.files_list_folder_continue(result.cursor)

    process_folder("", local_root)


workspace_root = find_workspace_root()
dataset_dir = workspace_root / "Dataset"
dataset_dir.mkdir(parents=True, exist_ok=True)


download_audio_from_shared_folder(
    SHARED_LINK,
    str(dataset_dir)
)


Listing shared folder path: ''
Listing shared folder path: '4'
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_1.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_10.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_100.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_101.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_102.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_103.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_104.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_105.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_106.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_107.wav
Verified existing file: c:\Users\alvco\Desktop\Manchester2026\Dataset\4\r_108.wav
Verified existing file: c:\Users\alvco